In [15]:
# Import required libraries

import pandas as pd
import numpy as np
import random
import re
import os

In [16]:
# Define project configuration

DATASET_PATH = "dataset.csv"

OUTPUT_DIR = "outputs"

CHUNK_SIZE = 500_000

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

In [17]:
# Normalize Persian and Arabic characters

def normalize_text(text):

    if pd.isna(text):
        return ""

    text = str(text)

    text = text.replace("ي", "ی")
    text = text.replace("ك", "ک")

    return text.strip()

# Question 1

Display ten random records from the dataset.

In [18]:
# Display ten random records

sample = pd.read_csv(
    DATASET_PATH,
    skiprows=lambda i:
    i > 0 and random.random() > 0.00005
)

sample.sample(10)

,national_code,FULL_NAME,FATHER_NAME,BIRTH_DATE,CITY_NAME,PROVINCE_NAME,BIRTH_CITY,BIRTH_PROVINCE
259,5259472,هاجر تميزكار,غلامعلي,1353-10-23,رشت,گيلان,رشت,گیلان
770,15655779,محمدسميع حقيقت شعار,جليل,1375-08-28,NaN,NaN,کرمانشاه,کرمانشاه
759,15425366,منا جعفري,محمد,1368-08-22,بندرعباس,هرمزگان,بندرعباس,هرمزگان
274,5564956,رويماه سادات شهامي تولون,ميرآقا,1342-04-04,تهران,تهران,مغان,اردبیل
1285,25661518,سيروس نوربخش شاهملكي,رحمت,1346-12-21,NaN,NaN,کرمانشاه,کرمانشاه
403,8250473,بتول سليماني پرلري,حسنقلي,1323-08-10,آذرشهر,اذربايجان شرقي,آذرشهر,آذربایجان شرقی
518,10821396,عامر عساكره,حميد,1361-01-20,NaN,NaN,شادگان,خوزستان
1176,23601961,سميرا رزمجوئي,اله حسين,1372-06-23,شيراز,فارس,شیراز,فارس
589,12228693,علي دورقي,مزعل,1361-10-01,اهواز,خوزستان,بندرماهشهر,خوزستان
1585,31367787,ساعد زماني سيبني,پرويز,1361-04-13,گيلان - رودبار,گيلان,NaN,NaN


# Question 2

Fill missing residence information using birth province and province capital city.

In [19]:
# Define province capitals

province_capitals = {
    "تهران": "تهران",
    "اصفهان": "اصفهان",
    "فارس": "شیراز",
    "مازندران": "ساری",
    "گیلان": "رشت",
    "مرکزی": "اراک",
    "هرمزگان": "بندرعباس",
    "خوزستان": "اهواز",
    "کرمان": "کرمان",
    "لرستان": "خرم آباد",
    "بوشهر": "بوشهر",
    "کرمانشاه": "کرمانشاه",
    "ایلام": "ایلام",
    "اردبیل": "اردبیل",
    "البرز": "کرج",
    "خراسان رضوی": "مشهد",
    "خراسان شمالی": "بجنورد",
    "آذربایجان شرقی": "تبریز",
    "آذربایجان غربی": "ارومیه"
}

In [20]:
# Generate Q2 output file

input_file = DATASET_PATH

output_file = (
    f"{OUTPUT_DIR}/Q2_output.csv"
)

first_chunk = True

for chunk in pd.read_csv(
    input_file,
    chunksize=CHUNK_SIZE
):

    missing_mask = (
        chunk["PROVINCE_NAME"]
        .isna()
    )

    chunk.loc[
        missing_mask,
        "PROVINCE_NAME"
    ] = chunk.loc[
        missing_mask,
        "BIRTH_PROVINCE"
    ]

    chunk.loc[
        missing_mask,
        "CITY_NAME"
    ] = (
        chunk.loc[
            missing_mask,
            "BIRTH_PROVINCE"
        ]
        .map(province_capitals)
    )

    chunk.to_csv(
        output_file,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )

    first_chunk = False

print(
    "Q2 completed successfully."
)

Q2 completed successfully.


# Question 3

Convert all national codes to a fixed 10-digit format.

In [21]:
# Generate Q3 output file

input_file = (
    f"{OUTPUT_DIR}/Q2_output.csv"
)

output_file = (
    f"{OUTPUT_DIR}/Q3_output.csv"
)

first_chunk = True

for chunk in pd.read_csv(
    input_file,
    chunksize=CHUNK_SIZE
):

    chunk["national_code"] = (
        chunk["national_code"]
        .astype(str)
        .str.zfill(10)
    )

    chunk.to_csv(
        output_file,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )

    first_chunk = False

print(
    "Q3 completed successfully."
)

Q3 completed successfully.


# Question 4

Find all customers with the last name Alizadeh living in Hormozgan province.

In [22]:
# Search for Alizadeh family members in Hormozgan

matches = []

for chunk in pd.read_csv(
    f"{OUTPUT_DIR}/Q3_output.csv",
    chunksize=CHUNK_SIZE
):

    names = (
        chunk["FULL_NAME"]
        .astype(str)
        .apply(normalize_text)
    )

    provinces = (
        chunk["PROVINCE_NAME"]
        .astype(str)
        .apply(normalize_text)
    )

    mask = (
        names.str.contains(
            "علیزاده",
            na=False
        )
    ) & (
        provinces.str.contains(
            "هرمزگان",
            na=False
        )
    )

    matches.append(
        chunk.loc[mask]
    )

result_q4 = pd.concat(
    matches,
    ignore_index=True
)

print(
    f"Total matches: {len(result_q4)}"
)

result_q4.head(20)

Total matches: 564


,national_code,FULL_NAME,FATHER_NAME,BIRTH_DATE,CITY_NAME,PROVINCE_NAME,BIRTH_CITY,BIRTH_PROVINCE
0,27291,سعيد عليزاده,فريدون,1363-06-29,بندرعباس,هرمزگان,بندرعباس,هرمزگان
1,36467,اميرعباس عليزاده,علي,1385-10-14,بندرعباس,هرمزگان,بندرعباس,هرمزگان
2,46244,حجت اله عليزاده فني,خداداد,1362-09-20,بستک,هرمزگان,پلدختر,لرستان
3,74782,زيور عليزاده حامد,قدرت,1367-06-12,بندرعباس,هرمزگان,عنبرآباد,کرمان
4,115490,كبري حاجي عليزاده,مريد,1358-01-01,ميناب,هرمزگان,شهربابک,کرمان
5,130116,معراج عليزاده,اسلام,1384-02-01,ميناب,هرمزگان,میناب,هرمزگان
6,196205,محمد عليزاده طولي,علي,1370-01-04,قشم,هرمزگان,قشم,هرمزگان
7,252801,محمدامين عليزاده اثفقن سري,موسي,1384-05-03,بندرعباس,هرمزگان,بندرعباس,هرمزگان
8,353386,فرزاد عليزاده قشمي,محمد,1370-12-01,قشم,هرمزگان,قشم,هرمزگان
9,365868,فرزاد عليزاده,خدارحم,1364-06-30,بندرعباس,هرمزگان,جاسک,هرمزگان


# Question 5

Create a new ADDRESS column and assign Aminian Dormitory as the residence address of Hassanpour family members living in Isfahan province.

In [ ]:
# Generate Q5 output file

input_file = (
    f"{OUTPUT_DIR}/Q3_output.csv"
)

output_file = (
    f"{OUTPUT_DIR}/Q5_output.csv"
)

first_chunk = True

total_updated = 0

for chunk in pd.read_csv(
    input_file,
    chunksize=CHUNK_SIZE
):

    # Create ADDRESS column if it does not exist

    if "ADDRESS" not in chunk.columns:
        chunk["ADDRESS"] = ""

    # Normalize text fields

    names = (
        chunk["FULL_NAME"]
        .astype(str)
        .apply(normalize_text)
    )

    provinces = (
        chunk["PROVINCE_NAME"]
        .astype(str)
        .apply(normalize_text)
    )

    # Find Hassanpour family members living in Isfahan

    mask = (
        names.str.contains(
            "حسنپور|حسن پور",
            na=False
        )
    ) & (
        provinces.str.contains(
            "اصفهان",
            na=False
        )
    )

    # Update address

    chunk.loc[
        mask,
        "ADDRESS"
    ] = "خوابگاه امینیان"

    total_updated += mask.sum()

    # Save processed chunk

    chunk.to_csv(
        output_file,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )

    first_chunk = False

print(
    f"Updated records: {total_updated}"
)

print(
    f"Output file saved to: {output_file}"
)

Updated records: 3371
Output file saved to: outputs/Q5_output.csv


In [ ]:
# Verify Question 5 output

q5_sample = pd.read_csv(
    f"{OUTPUT_DIR}/Q5_output.csv",
    usecols=[
        "FULL_NAME",
        "PROVINCE_NAME",
        "ADDRESS"
    ],
    nrows=100000
)

q5_sample[
    q5_sample["ADDRESS"]
    == "خوابگاه امینیان"
].head(20)